# 补跑 `sft_rag` 路线（只生成，不训练）

上一轮已经把 LoRA 训好并打包在 `lixiang_results.zip` 里了。
这个 notebook 复用那个适配器，只补跑第四条路线：

> **`sft_rag` = 微调后的模型 + 检索到的财报原文**

口径来自微调，事实来自检索 —— 四条路线里唯一三条腿都齐的一条。

**全程不训练**，只做生成，T4 上约 25–30 分钟。

### 跑之前

右上角 `Runtime` → `Change runtime type` → 确认是 **T4 GPU**。

> 免费版 Colab 的 GPU 配额是黑盒，用完会报「已达到使用量限额」且不说何时恢复。
> 真遇到了就等 12–24 小时再跑，代码本身没问题。

## 1 · 安装依赖

In [ ]:
# unsloth 在 Colab 上一键装好，装不上也无所谓 —— 加载模型时会自动退回纯 peft
!pip install -q unsloth 2>/dev/null || echo "unsloth 装不上，将使用 peft 回退方案"
!pip install -q transformers peft datasets accelerate bitsandbytes

## 2 · 上传数据 + 适配器

点下面这格的运行按钮，会弹出「选择文件」——
选本地的 **`train/lixiang_gen_bundle.zip`**（约 77 MB）。

> 是**点按钮选文件**，不是拖拽。上传 77 MB 要等几分钟，进度条走完再往下跑。

In [ ]:
import os, zipfile

if not os.path.exists('/content/lixiang/lora_adapter'):
    from google.colab import files
    # 点「选择文件」，选 lixiang_gen_bundle.zip（约 77 MB，上传要等几分钟）
    up = files.upload()
    name = list(up)[0]
    with zipfile.ZipFile(name) as z:
        z.extractall('/content')

os.system('find /content/lixiang -type f | head -20')
DATA, EVAL = '/content/lixiang/data', '/content/lixiang/eval'
ADAPTER = '/content/lixiang/lora_adapter'
assert os.path.exists(f'{ADAPTER}/adapter_model.safetensors'), \
    '适配器没解压出来 —— 确认上传的是 lixiang_gen_bundle.zip'
print('\n数据 + 适配器就位 ✓')

## 3 · 先试跑 3 条（约 3 分钟）

**别跳过这格。** 跑全量要 25 分钟，先花 3 分钟确认三件事都搭对了：

1. 适配器挂上了吗（回答应该是董秘口吻，不是「我无法提供」）
2. 检索通了吗（回答里的数字应该能在原文片段里找到）
3. 输出格式对吗（应该有【依据：《…》第 N 页】）

三条里有任何一条不对，就**别往下跑**，先回来找我。

第一次跑会向量化 1431 页（1–2 分钟）并把结果缓存下来，
第 4 格会直接读缓存，不会重复算。

In [ ]:
!python /content/lixiang/eval/generate.py --arm sft_rag \
    --adapter /content/lixiang/lora_adapter --load-4bit --limit 3 --out /content/preflight

import json
print('\n' + '=' * 78)
for line in open('/content/preflight/sft_rag.jsonl', encoding='utf-8'):
    r = json.loads(line)
    print('-' * 78)
    print(f"{r['cell']}  {r['qa_id']}")
    print(r['prediction'][:500])

## 4 · 跑满 140 条

上面三条看着没问题就往下跑。约 20–25 分钟，会打印进度。

跑完结果落在 `/content/lixiang/eval/preds/sft_rag.jsonl`。

In [ ]:
!python /content/lixiang/eval/generate.py --arm sft_rag \
    --adapter /content/lixiang/lora_adapter --load-4bit

# 顺手自检：条数对不对、有没有出现复读
import json
p = '/content/lixiang/eval/preds/sft_rag.jsonl'
rows = [json.loads(l) for l in open(p, encoding='utf-8')]
print(f'\n{len(rows)} 条')
assert len(rows) == 140, f'应该是 140 条，实际 {len(rows)} 条 —— 别下载，先告诉我'
print('条数正确 ✓')
print('\n前两条长这样：')
for r in rows[:2]:
    print('-' * 78)
    print(f"{r['cell']}  {r['qa_id']}")
    print(r['prediction'][:300])

## 5 · 打包下载

In [ ]:
!cd /content && zip -q lixiang_sft_rag.zip lixiang/eval/preds/sft_rag.jsonl
!ls -la /content/lixiang_sft_rag.zip

from google.colab import files
files.download('/content/lixiang_sft_rag.zip')

## 跑完之后

把 `lixiang_sft_rag.zip` 里的 `sft_rag.jsonl` 放到本地 `eval/preds/` 下，
然后我在本地跑判分，四条路线一起出四格对照表。

**Colab 没有「暂停」**——关标签页 = 虚拟机销毁，训练结果和预测全没。
跑的时候别关标签页。